# Figure S2E — 1-Year Cumulative Incidence by Line of Therapy + Shared-Frailty Cox Models

Self-contained notebook, two parts:

**Part 1 — Descriptive:** KM-based 1-year cumulative incidence per toxicity, stratified by line of
therapy (LOT 1 / 2 / 3 / 4+), with 95% CI error bars. Each `(patient, LOT)` pair is one observation;
`T=0` = that line's start; censoring = `min(lot_end+180d, next_lot_start, death, last_fu)`. LOT
groups are compared with **multivariable** Cox proportional-hazards models (adjusted for age, sex,
and cancer type).

**Part 2 — Inferential:** A shared Gamma-frailty Cox proportional-hazards model per toxicity, testing
whether line of therapy independently predicts hazard of toxicity while accounting for patient-level
susceptibility (frailty). Fixed effects: line of therapy (ref = LOT 1), age at line start, sex, and
cancer type (top 10 + Other). Random effect: patient-specific frailty term.

**Why a custom implementation:** there is no off-the-shelf Python equivalent of R's `coxme`/`frailty()`.
Part 2 implements the classic Gamma-frailty EM algorithm (Klein, 1992) directly on top of a
Breslow-tie Cox partial likelihood — no R dependency. This was validated on simulated clustered
survival data with known parameters before being applied here (recovered known `beta`/frailty-variance
to within simulation noise).

**Outputs:**
- `LOT_Prevalence_S2E.pdf` — descriptive bar chart
- `LOT_Prevalence_S2E_cumulative_incidence.csv` — 1-year CI / 95% CI values per LOT group × toxicity
- `LOT_Prevalence_S2E_frailty_cox_results.csv` — hazard ratios, 95% CI, p-values, and frailty variance
  (theta) per toxicity model

**Formatting (Nature compliance):** Arial only (hard-fails if not resolved), `pdf.fonttype=42`,
7pt axis labels / 6pt tick labels / 5pt legend, no `bbox_inches='tight'` on save.

In [ ]:
import os
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.rcParams['font.family'] = 'sans-serif'
matplotlib.rcParams['font.sans-serif'] = ['Arial']
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42

import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from lifelines import KaplanMeierFitter
from scipy.optimize import minimize, minimize_scalar
from scipy.special import digamma, gammaln
from scipy.stats import norm

%matplotlib inline

warnings.filterwarnings('ignore')

# ---- Hard-fail if Arial isn't actually resolved (no silent fallback) ----
import matplotlib.font_manager as fm
_arial_path = fm.findfont('Arial', fallback_to_default=False)
if 'Arial' not in _arial_path:
    raise RuntimeError(
        f"Arial not found -- matplotlib resolved to '{_arial_path}' instead. "
        "Install Arial or update font.sans-serif before rendering this figure."
    )
print(f"Arial resolved to: {_arial_path}")


## Paths

Notebook lives in `figure 2/scripts/`. Data lives in the sibling `figure 2/data/` folder; outputs go to
`figure 2/results/supp/S2E_LOT_Prevalence/` (rename below if a different folder name is preferred).


In [ ]:
NOTEBOOK_DIR = os.getcwd()
FIGURES_DIR = os.path.normpath(os.path.join(NOTEBOOK_DIR, '..', '..', '..'))
DATA_DIR = os.path.join(FIGURES_DIR, 'figures_data', 'figure 2', 'data')
COVARS_PATH = os.path.join(DATA_DIR, 'OneDrive_1_8-7-2026', 'llm84k_pneumonitis_grade0_20260630.csv')
RESULTS_DIR = os.path.normpath(os.path.join(NOTEBOOK_DIR, '..', 'results', 'supp', 'S2E_LOT_Prevalence'))
os.makedirs(RESULTS_DIR, exist_ok=True)

LLM_PATIENT_PATH = os.path.join(DATA_DIR, 'llm_calls_patient_level_84k.csv')
LLM_BATCH_PATH = os.path.join(DATA_DIR, 'llm_calls_batch_level_84k.csv')

PDF_OUT = os.path.join(RESULTS_DIR, 'LOT_Prevalence_S2E.pdf')
CSV_OUT = os.path.join(RESULTS_DIR, 'LOT_Prevalence_S2E_cumulative_incidence.csv')
FRAILTY_CSV_OUT = os.path.join(RESULTS_DIR, 'LOT_Prevalence_S2E_frailty_cox_results.csv')

for p in [COVARS_PATH, LLM_PATIENT_PATH, LLM_BATCH_PATH]:
    print(('FOUND   ' if os.path.exists(p) else 'MISSING '), p)

## Constants


In [ ]:
TOXICITY_COLUMNS = ['liver_toxicity', 'hypothyroidism', 'pneumonitis',
                    'colitis', 'adrenal_insufficiency', 'hyperthyroidism']

TOXICITY_DISPLAY = {
    'pneumonitis': 'Pneumonitis', 'adrenal_insufficiency': 'Adrenal Insufficiency',
    'liver_toxicity': 'Liver Toxicity', 'colitis': 'Colitis',
    'hyperthyroidism': 'Hyperthyroidism', 'hypothyroidism': 'Hypothyroidism',
}

LOT_GROUPS = ['1', '2', '3', '4+']
LOT_GROUP_COLORS = {'1': '#3498DB', '2': '#2ECC71', '3': '#E67E22', '4+': '#E74C3C'}

T_MONTHS = 12.0

def standardize_mrn(mrn):
    if pd.isna(mrn):
        return None
    try:
        digits = re.findall(r'\d+', str(mrn).strip().strip("'\""))
        return str(int(digits[0])).zfill(8) if digits else None
    except (ValueError, TypeError):
        return None


## Build `all_lots` — every line of therapy for every patient

Unlike the age/sex panels (which use only the first LOT), this panel needs *every* LOT row per
patient, each with its own start time and censoring window (`T=0` = that line's start). Censoring
per line: `min(lot_end + 180d, next_lot_start, death, last_fu)`.


In [ ]:
covars = pd.read_csv(COVARS_PATH, encoding='latin-1', low_memory=False)
covars['mrn'] = covars['mrn'].apply(standardize_mrn)
covars = covars[covars['mrn'].notna()].copy()

covars['lot'] = pd.to_numeric(covars['lot'], errors='coerce')
covars['age_at_lot_start'] = pd.to_numeric(covars['age_at_lot_start'], errors='coerce')
for dcol in ['lot_start', 'lot_end', 'dod', 'last_fu']:
    if dcol in covars.columns:
        covars[dcol] = pd.to_datetime(covars[dcol], errors='coerce')

covars = covars.sort_values(['mrn', 'lot'])

# ---- restrict to patients with LLM predictions (cohort membership only) ----
llm_patients = pd.read_csv(LLM_PATIENT_PATH, encoding='latin-1', low_memory=False)
llm_patients['mrn'] = llm_patients['mrn'].apply(standardize_mrn)
llm_patients = llm_patients[llm_patients['mrn'].notna()].copy()
for tox in TOXICITY_COLUMNS:
    if tox in llm_patients.columns:
        llm_patients[tox] = pd.to_numeric(llm_patients[tox], errors='coerce').fillna(0).astype(int)
print(f'{len(llm_patients):,} patients with LLM predictions')

all_lots = covars.merge(llm_patients[['mrn'] + TOXICITY_COLUMNS], on='mrn', how='inner', suffixes=('', '_ae'))
print(f'{len(all_lots):,} LOT-level records for {all_lots["mrn"].nunique():,} patients')

# ---- per-line censoring ----
all_lots['lot_clean'] = pd.to_numeric(all_lots['lot'], errors='coerce')
all_lots = all_lots[all_lots['lot_clean'].notna() & all_lots['lot_start'].notna()].copy()
all_lots = all_lots.sort_values(['mrn', 'lot_clean'])
all_lots['next_lot_start'] = all_lots.groupby('mrn')['lot_start'].shift(-1)

def censor_at(row):
    cands = []
    if pd.notna(row.get('lot_end')):
        cands.append(row['lot_end'] + pd.Timedelta(days=180))
    if pd.notna(row.get('next_lot_start')):
        cands.append(row['next_lot_start'])
    if pd.notna(row.get('dod')):
        cands.append(row['dod'])
    if pd.notna(row.get('last_fu')):
        cands.append(row['last_fu'])
    if not cands:
        return np.nan
    return max(0, (min(cands) - row['lot_start']).days)

all_lots['censor_days'] = all_lots.apply(censor_at, axis=1)
all_lots = all_lots[all_lots['censor_days'].notna() & (all_lots['censor_days'] > 0)].copy()

all_lots['lot_group'] = all_lots['lot_clean'].clip(upper=4).astype(int).astype(str)
all_lots.loc[all_lots['lot_clean'] >= 4, 'lot_group'] = '4+'

print(all_lots['lot_group'].value_counts().reindex(LOT_GROUPS))


## Load batch-level toxicity data


In [ ]:
batch_df = pd.read_csv(LLM_BATCH_PATH, encoding='latin-1', low_memory=False)
batch_df['mrn'] = batch_df['mrn'].apply(standardize_mrn)
batch_df = batch_df[batch_df['mrn'].notna()].copy()
batch_df['window_start'] = pd.to_datetime(batch_df['window_start'], errors='coerce')
batch_df['window_end'] = pd.to_datetime(batch_df['window_end'], errors='coerce')
batch_df = batch_df.dropna(subset=['window_start', 'window_end'])
print(f'{len(batch_df):,} batch records for {batch_df["mrn"].nunique():,} patients')

## Part 1 — KM 1-year cumulative incidence per LOT group

For a given LOT-group cohort (subset of `all_lots` rows) and toxicity: find each row's own first-AE
time within its `[lot_start, lot_start + censor_days]` window, fit a Kaplan-Meier curve, and read off
the cumulative incidence (1 − survival) and its 95% CI at 12 months.


In [ ]:
def ci_at_t_for_lot(lot_rows, tox):
    if len(lot_rows) < 10 or tox not in batch_df.columns:
        return 0.0, 0.0, 0.0
    bsub = batch_df[batch_df[tox] == 1][['mrn', 'window_start']].copy()
    bsub_by_mrn = dict(list(bsub.groupby('mrn')))
    times, events = [], []
    for row in lot_rows.itertuples():
        mrn, lot_start, censor_d = row.mrn, row.lot_start, row.censor_days
        first_ae_days = None
        if mrn in bsub_by_mrn:
            recs = bsub_by_mrn[mrn]
            days = (recs['window_start'] - lot_start).dt.days
            in_window = days[(days >= 0) & (days <= censor_d)]
            if len(in_window) > 0:
                first_ae_days = float(in_window.min())
        if first_ae_days is not None:
            times.append(first_ae_days); events.append(1)
        else:
            times.append(censor_d); events.append(0)
    times, events = np.array(times), np.array(events)
    events = events[times > 0]
    times = times[times > 0]
    if len(times) < 10:
        return 0.0, 0.0, 0.0
    surv = pd.DataFrame({'time_months': times / 30.44, 'event': events})
    kmf = KaplanMeierFitter()
    kmf.fit(surv['time_months'], surv['event'])
    ci = (1 - kmf.survival_function_at_times(T_MONTHS).values[0]) * 100
    ci_tbl = kmf.confidence_interval_survival_function_
    idx = max(0, min(np.searchsorted(kmf.survival_function_.index, T_MONTHS, side='right') - 1,
                     len(ci_tbl) - 1))
    lo = (1 - ci_tbl.iloc[idx, 1]) * 100
    hi = (1 - ci_tbl.iloc[idx, 0]) * 100
    return ci, lo, hi


In [ ]:
km_records = []
lot_group_rows = {}

for grp in LOT_GROUPS:
    grp_rows = all_lots[all_lots['lot_group'] == grp]
    lot_group_rows[grp] = grp_rows
    if len(grp_rows) < 10:
        continue
    for tox in TOXICITY_COLUMNS:
        pct, lo, hi = ci_at_t_for_lot(grp_rows, tox)
        km_records.append({
            'lot_group': grp,
            'n_lines': len(grp_rows),
            'toxicity': tox,
            'toxicity_display': TOXICITY_DISPLAY[tox],
            'cumulative_incidence_pct': round(pct, 3),
            'ci_95_lower': round(lo, 3),
            'ci_95_upper': round(hi, 3),
        })

km_results_df = pd.DataFrame(km_records)
km_results_df.to_csv(CSV_OUT, index=False)
print(f'Saved: {os.path.basename(CSV_OUT)}')
km_results_df


## Pairwise Cox PH: LOT 1 (reference) vs. each other LOT group, per toxicity (Multivariable)

Every other line-of-therapy group is compared against the **same fixed reference, LOT 1**:
LOT 1 vs. LOT 2, LOT 1 vs. LOT 3, LOT 1 vs. LOT 4+. Each comparison is a **multivariable** Cox
proportional-hazards model matching the covariate adjustment used in the main figure panels (2C, 2D, 2E).

The model includes:
- `group`: binary (0 = LOT 1 reference, 1 = comparator LOT) — the exposure of interest
- `has_pd1_flag`, `has_ctla4_flag`: binary treatment flags (combined from raw columns)
- `contains_chemo`, `contains_hormone`, `contains_biologic`, `contains_targeted`: binary treatment flags
- `age_centered`: continuous, mean-centered age at LOT start
- `sex_female`: binary (0 = Male, 1 = Female)
- `cancer_type_*`: dummy variables for cancer type (largest category as reference)

A two-sample log-rank test (unadjusted) on the same data is computed as an independent cross-check.
Multiplicity-adjusted (BH-FDR, Bonferroni) p-values across the 6x3=18 tests are also computed.

**Important caveat -- not the same model as Part 2.** This pairwise test treats each
`(patient, LOT)` row as an independent observation, exactly like the KM curves it annotates --
it does **not** account for the same patient contributing multiple LOT rows (e.g. one patient's
LOT1, LOT2, and LOT3 rows are all treated as independent here). That within-patient correlation
is precisely what the shared-frailty model in **Part 2** is built to handle. Both now use LOT 1
as the reference, but Part 2 uses a more rigorous estimator with patient-level frailty -- the two
p-values answer related but distinct questions and are not expected to match exactly. Requires
>=10 rows per arm and >=5 pooled events; otherwise NaN.


In [ ]:
from lifelines import CoxPHFitter
from lifelines.statistics import logrank_test

# ============================================================================
# Prepare covariate data for multivariable model
# Matches the adjustment covariates used in main figure panels (2C, 2D, 2E)
# ============================================================================

def _flag_on(x):
    """Convert various flag formats to boolean."""
    if pd.isna(x):
        return False
    if isinstance(x, bool):
        return x
    if isinstance(x, (int, float)):
        return x == 1
    return str(x).strip().lower() in ('1', 'true', 'yes')

# Build covariate dataframe with all adjustment variables (LOT level)
covar_cols = ['mrn', 'age_at_lot_start', 'sex', 'cancer_type',
              'contains_ctla4_immuno', 'contains_ctla4', 'contains_non_ctla4_immuno', 'contains_pd1',
              'contains_chemo', 'contains_hormone', 'contains_biologic', 'contains_targeted']
covar_df_lot = all_lots[[c for c in covar_cols if c in all_lots.columns]].copy()
covar_df_lot['age_at_lot_start'] = pd.to_numeric(covar_df_lot['age_at_lot_start'], errors='coerce')

# Clean sex
covar_df_lot['sex_clean'] = covar_df_lot['sex'].astype(str).str.strip().str.capitalize()
covar_df_lot['sex_female'] = (covar_df_lot['sex_clean'] == 'Female').astype(float)
covar_df_lot.loc[~covar_df_lot['sex_clean'].isin(['Male', 'Female']), 'sex_female'] = np.nan

# Combined immuno flags (same logic as panel 2E)
covar_df_lot['has_ctla4_flag'] = (covar_df_lot['contains_ctla4_immuno'].apply(_flag_on) |
                                   covar_df_lot['contains_ctla4'].apply(_flag_on)).astype(int)
covar_df_lot['has_pd1_flag'] = (covar_df_lot['contains_non_ctla4_immuno'].apply(_flag_on) |
                                 covar_df_lot['contains_pd1'].apply(_flag_on)).astype(int)

# Treatment flags
for col in ['contains_chemo', 'contains_hormone', 'contains_biologic', 'contains_targeted']:
    if col in covar_df_lot.columns:
        covar_df_lot[col] = covar_df_lot[col].apply(_flag_on).astype(int)
    else:
        covar_df_lot[col] = 0

# Clean cancer_type
covar_df_lot['cancer_type'] = covar_df_lot['cancer_type'].fillna('Unknown').astype(str).str.strip()
covar_df_lot.loc[covar_df_lot['cancer_type'] == '', 'cancer_type'] = 'Unknown'
cancer_type_counts = covar_df_lot['cancer_type'].value_counts()
CANCER_TYPE_REF = cancer_type_counts.index[0]

ADJUST_COLS = ['has_pd1_flag', 'has_ctla4_flag', 'contains_chemo', 'contains_hormone',
               'contains_biologic', 'contains_targeted']
print(f"Adjustment covariates: {ADJUST_COLS} + age + sex + cancer_type")
print(f"Cancer type reference (most common): {CANCER_TYPE_REF} (n={cancer_type_counts.iloc[0]:,})")


def _build_pairwise_surv_lot(ref_rows, cmp_rows, tox):
    """Time-to-first-AE survival table for two LOT-group row subsets, with a binary `group`
    indicator (0 = ref_rows, 1 = cmp_rows) and covariates for multivariable modeling."""
    if tox not in batch_df.columns:
        return None
    combined = pd.concat([
        ref_rows.assign(_pw_group=0),
        cmp_rows.assign(_pw_group=1),
    ], ignore_index=True)
    if len(combined) < 20:
        return None
    bsub = batch_df[batch_df[tox] == 1][['mrn', 'window_start']]
    bsub_by_mrn = dict(list(bsub.groupby('mrn')))
    times, events = [], []
    for row in combined.itertuples():
        mrn, lot_start, censor_d = row.mrn, row.lot_start, row.censor_days
        first_ae_days = None
        if mrn in bsub_by_mrn:
            recs = bsub_by_mrn[mrn]
            days = (recs['window_start'] - lot_start).dt.days
            in_window = days[(days >= 0) & (days <= censor_d)]
            if len(in_window) > 0:
                first_ae_days = float(in_window.min())
        if first_ae_days is not None:
            times.append(first_ae_days); events.append(1)
        else:
            times.append(censor_d); events.append(0)
    combined = combined.assign(_pw_time=np.array(times), _pw_event=np.array(events))
    combined = combined[combined['_pw_time'] > 0].copy()
    if len(combined) < 20:
        return None
    combined['time_months'] = combined['_pw_time'] / 30.44
    combined = combined.rename(columns={'_pw_event': 'event', '_pw_group': 'group'})
    return combined


def cox_hr_pvalue_lot_multivariable(ref_rows, cmp_rows, tox):
    """Multivariable Cox PH: comparison LOT vs reference LOT, adjusting for treatment flags,
    age (centered), sex (binary), and cancer_type (dummy-encoded). Matches the covariate
    adjustment used in main figure panels."""
    n_ref_input, n_cmp_input = len(ref_rows), len(cmp_rows)
    if n_ref_input < 10 or n_cmp_input < 10:
        return np.nan, np.nan, np.nan, np.nan, n_ref_input, n_cmp_input, np.nan, np.nan
    
    surv = _build_pairwise_surv_lot(ref_rows, cmp_rows, tox)
    if surv is None:
        return np.nan, np.nan, np.nan, np.nan, n_ref_input, n_cmp_input, np.nan, np.nan
    
    # Add covariates from the original rows (already in surv from _build_pairwise_surv_lot)
    surv['age_at_lot_start'] = pd.to_numeric(surv['age_at_lot_start'], errors='coerce')
    surv['sex_clean'] = surv['sex'].astype(str).str.strip().str.capitalize()
    surv['sex_female'] = (surv['sex_clean'] == 'Female').astype(float)
    surv.loc[~surv['sex_clean'].isin(['Male', 'Female']), 'sex_female'] = np.nan
    surv['cancer_type'] = surv['cancer_type'].fillna('Unknown').astype(str).str.strip()
    
    # Combined immuno flags
    surv['has_ctla4_flag'] = (surv['contains_ctla4_immuno'].apply(_flag_on) |
                              surv['contains_ctla4'].apply(_flag_on)).astype(int)
    surv['has_pd1_flag'] = (surv['contains_non_ctla4_immuno'].apply(_flag_on) |
                            surv['contains_pd1'].apply(_flag_on)).astype(int)
    
    # Treatment flags
    for col in ['contains_chemo', 'contains_hormone', 'contains_biologic', 'contains_targeted']:
        if col in surv.columns:
            surv[col] = surv[col].apply(_flag_on).astype(int)
        else:
            surv[col] = 0
    
    # Drop rows with missing required covariates
    surv = surv.dropna(subset=['age_at_lot_start', 'sex_female', 'cancer_type'])
    
    if len(surv) < 20:
        return np.nan, np.nan, np.nan, np.nan, n_ref_input, n_cmp_input, np.nan, np.nan
    
    n_ref = int((surv['group'] == 0).sum())
    n_cmp = int((surv['group'] == 1).sum())
    events_ref = int(surv.loc[surv['group'] == 0, 'event'].sum())
    events_cmp = int(surv.loc[surv['group'] == 1, 'event'].sum())
    
    if surv['event'].sum() < 5 or surv['group'].nunique() < 2:
        return np.nan, np.nan, np.nan, np.nan, n_ref, n_cmp, events_ref, events_cmp
    
    # Center age
    surv['age_centered'] = surv['age_at_lot_start'] - surv['age_at_lot_start'].mean()
    
    # Dummy-encode cancer_type
    cancer_dummies = pd.get_dummies(surv['cancer_type'], prefix='cancer', drop_first=False)
    ref_col = f'cancer_{CANCER_TYPE_REF}'
    if ref_col in cancer_dummies.columns:
        cancer_dummies = cancer_dummies.drop(columns=[ref_col])
    
    # Build model dataframe with all covariates
    # Order: exposure (LOT group), treatment flags, age, sex, cancer_type dummies
    model_cols = ['time_months', 'event', 'group'] + ADJUST_COLS + ['age_centered', 'sex_female']
    model_df = pd.concat([surv[model_cols].reset_index(drop=True), 
                          cancer_dummies.reset_index(drop=True)], axis=1)
    
    # Fill any NaN in treatment flags with 0
    for col in ADJUST_COLS:
        model_df[col] = model_df[col].fillna(0).astype(int)
    
    cph = CoxPHFitter(penalizer=0.1, l1_ratio=0.0)
    try:
        cph.fit(model_df, duration_col='time_months', event_col='event')
    except Exception as e:
        print(f"  Cox fit failed for {tox}: {e}")
        return np.nan, np.nan, np.nan, np.nan, n_ref, n_cmp, events_ref, events_cmp
    
    hr = float(np.exp(cph.params_['group']))
    hr_lower = float(cph.summary.loc['group', 'exp(coef) lower 95%'])
    hr_upper = float(cph.summary.loc['group', 'exp(coef) upper 95%'])
    p = float(cph.summary.loc['group', 'p'])
    return hr, hr_lower, hr_upper, p, n_ref, n_cmp, events_ref, events_cmp


def logrank_pvalue_lot(ref_rows, cmp_rows, tox):
    surv = _build_pairwise_surv_lot(ref_rows, cmp_rows, tox)
    if surv is None:
        return np.nan
    is_cmp = surv['group'] == 1
    if is_cmp.sum() < 10 or (~is_cmp).sum() < 10:
        return np.nan
    try:
        lr = logrank_test(
            surv.loc[is_cmp, 'time_months'], surv.loc[~is_cmp, 'time_months'],
            event_observed_A=surv.loc[is_cmp, 'event'], event_observed_B=surv.loc[~is_cmp, 'event'],
        )
        return float(lr.p_value)
    except Exception:
        return np.nan


def bh_fdr(pvals):
    """Benjamini-Hochberg FDR-adjusted p-values (q-values), NaN-safe."""
    p = np.asarray(pvals, dtype=float)
    out = np.full_like(p, np.nan)
    valid_idx = np.where(~np.isnan(p))[0]
    m = len(valid_idx)
    if m == 0:
        return out
    vp = p[valid_idx]
    order = np.argsort(vp)
    ranked_p = vp[order]
    ranks = np.arange(1, m + 1)
    q = ranked_p * m / ranks
    q = np.minimum.accumulate(q[::-1])[::-1]
    q = np.clip(q, 0, 1)
    out[valid_idx[order]] = q
    return out


def bonferroni_adjust(pvals):
    """Bonferroni-adjusted p-values, NaN-safe."""
    p = np.asarray(pvals, dtype=float)
    out = np.full_like(p, np.nan)
    valid_idx = np.where(~np.isnan(p))[0]
    m = len(valid_idx)
    if m == 0:
        return out
    out[valid_idx] = np.clip(p[valid_idx] * m, 0, 1)
    return out


# Reference (control) group = LOT 1, fixed -- every other group is compared against it.
ref_group = '1'
cmp_groups = [g for g in LOT_GROUPS if g != ref_group]
print(f"\nReference LOT group (fixed): 'LOT {ref_group}' (N={len(lot_group_rows[ref_group]):,} rows)")
for g in cmp_groups:
    print(f"  comparator 'LOT {g}': N={len(lot_group_rows[g]):,} rows")

cox_records = []
for tox in TOXICITY_COLUMNS:
    print(f"Fitting multivariable Cox for: {tox}")
    for cmp_grp in cmp_groups:
        hr, hr_lo, hr_hi, p, n_ref, n_cmp, ev_ref, ev_cmp = cox_hr_pvalue_lot_multivariable(
            lot_group_rows[ref_group], lot_group_rows[cmp_grp], tox)
        lr_p = logrank_pvalue_lot(lot_group_rows[ref_group], lot_group_rows[cmp_grp], tox)
        cox_records.append({
            'toxicity': tox,
            'toxicity_display': TOXICITY_DISPLAY[tox],
            'reference_group': f'LOT {ref_group}',
            'comparison_group': f'LOT {cmp_grp}',
            'n_reference': n_ref,
            'n_comparison': n_cmp,
            'events_reference': ev_ref,
            'events_comparison': ev_cmp,
            'hazard_ratio': round(hr, 3) if pd.notna(hr) else np.nan,
            'hr_lower_95': round(hr_lo, 3) if pd.notna(hr_lo) else np.nan,
            'hr_upper_95': round(hr_hi, 3) if pd.notna(hr_hi) else np.nan,
            'p_value_cox': p,
            'p_value_logrank_unadj': lr_p,
        })

cox_df = pd.DataFrame(cox_records)
raw_p = cox_df['p_value_cox'].values
cox_df['p_value_fdr_bh'] = bh_fdr(raw_p)
cox_df['p_value_bonferroni'] = bonferroni_adjust(raw_p)

# Merge onto km_results_df (comparator rows only; the LOT4+ reference row has nothing to compare
# itself to, so it gets NaN) and re-save the CSV.
pmap = cox_df.set_index(['toxicity', 'comparison_group'])[
    ['hazard_ratio', 'hr_lower_95', 'hr_upper_95', 'p_value_cox', 'p_value_logrank_unadj', 'p_value_fdr_bh', 'p_value_bonferroni']]
km_results_df['lot_group_label'] = 'LOT ' + km_results_df['lot_group']
km_results_df = km_results_df.merge(
    pmap, left_on=['toxicity', 'lot_group_label'], right_index=True, how='left'
)
km_results_df = km_results_df.drop(columns=['lot_group_label']).rename(columns={
    'hazard_ratio': 'hr_vs_lot1_adj',
    'hr_lower_95': 'hr_vs_lot1_adj_lower_95',
    'hr_upper_95': 'hr_vs_lot1_adj_upper_95',
    'p_value_cox': 'p_value_vs_lot1_adj',
    'p_value_logrank_unadj': 'p_value_logrank_vs_lot1_unadj',
    'p_value_fdr_bh': 'p_value_vs_lot1_adj_fdr_bh',
    'p_value_bonferroni': 'p_value_vs_lot1_adj_bonferroni',
})
km_results_df['reference_group'] = f'LOT {ref_group}'

km_results_df.to_csv(CSV_OUT, index=False)
print(f'\nSaved (with multivariable Cox + unadjusted log-rank p-values): {os.path.basename(CSV_OUT)}')
print(f'Adjustments: age (centered), sex, cancer_type (reference: {CANCER_TYPE_REF})')
cox_df[['toxicity_display', 'comparison_group', 'hazard_ratio', 'p_value_cox', 'p_value_fdr_bh', 'p_value_bonferroni']]

## Plot


In [ ]:
def set_axes_position_inches(fig, ax, left_in, top_in, width_in, height_in):
    fw, fh = fig.get_size_inches()
    ax.set_position([
        left_in / fw,
        1 - (top_in + height_in) / fh,
        width_in / fw,
        height_in / fh,
    ])

FIG_WIDTH_IN  = 3.6
FIG_HEIGHT_IN = 2.3

In [ ]:
PLOT_ADJUSTED_P = 'cox'  # 'cox' (raw), 'fdr_bh', or 'bonferroni'
P_COL = {'cox': 'p_value_cox', 'fdr_bh': 'p_value_fdr_bh', 'bonferroni': 'p_value_bonferroni'}[PLOT_ADJUSTED_P]

def format_pval(p):
    if pd.isna(p):
        return ''
    if p < 0.001:
        return 'p < 0.001'
    return f'p = {p:.3f}'

ae_list = TOXICITY_COLUMNS
display_names = [TOXICITY_DISPLAY[t] for t in ae_list]
groups = LOT_GROUPS
colors = LOT_GROUP_COLORS
n_ae = len(ae_list)
n_levels = len(cmp_groups)  # stacked p-value brackets per toxicity

fig, ax = plt.subplots(figsize=(FIG_WIDTH_IN, FIG_HEIGHT_IN))
total_width = 0.80
bar_width = total_width / len(groups)
x = np.arange(n_ae)

bar_x_pos = {}
for j, grp in enumerate(groups):
    grp_rows = lot_group_rows[grp]
    bar_off = -total_width / 2 + (j + 0.5) * bar_width
    bar_x_pos[grp] = bar_off
    if len(grp_rows) < 10:
        continue
    sub = km_results_df[km_results_df['lot_group'] == grp].set_index('toxicity').loc[ae_list]
    g_pct = sub['cumulative_incidence_pct'].to_numpy()
    g_lo = sub['ci_95_lower'].to_numpy()
    g_hi = sub['ci_95_upper'].to_numpy()
    ax.bar(x + bar_off, g_pct, bar_width, yerr=[g_pct - g_lo, g_hi - g_pct], capsize=1.5,
           color=colors[grp], edgecolor='none', label=f'LOT {grp} (N={len(grp_rows):,})',
           ecolor='gray', error_kw={'linewidth': 0.5})

ax.set_xlabel('Adverse event', fontsize=7)
ax.set_ylabel('1-year cumulative\nincidence (%, 95% CI)', fontsize=7)
ax.set_xticks(x)
ax.set_xticklabels(display_names, rotation=30, ha='right', fontsize=6)
ax.tick_params(axis='y', labelsize=6)
ax.legend(fontsize=5, framealpha=0.95, ncol=4, loc='upper center', bbox_to_anchor=(0.5, 1.15),
          handlelength=1, columnspacing=0.8)

# ---- reserve extra headroom above the bars for stacked p-value brackets ----
data_max = ax.get_ylim()[1]
pad_frac = 0.15 + 0.14 * n_levels
new_top = min(data_max * (1 + pad_frac), 60)
ax.set_ylim(0, new_top)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# ---- p-value brackets: LOT4+ (reference) vs. each comparator LOT group, per toxicity ----
p_lookup = cox_df.set_index(['toxicity', 'comparison_group'])[P_COL]
y_range = ax.get_ylim()[1]
tick_h = y_range * 0.02
level_gap = y_range * 0.11

for i, tox in enumerate(ae_list):
    tops_all = km_results_df.loc[km_results_df['toxicity'] == tox, 'ci_95_upper']
    base_top = tops_all.max() if len(tops_all) else 0
    for lvl, cmp_grp in enumerate(cmp_groups):
        key = (tox, f'LOT {cmp_grp}')
        if key not in p_lookup.index:
            continue
        label = format_pval(p_lookup.loc[key])
        if not label:
            continue
        x_left = x[i] + bar_x_pos[ref_group]
        x_right = x[i] + bar_x_pos[cmp_grp]
        if x_left > x_right:
            x_left, x_right = x_right, x_left
        y_bar = base_top + level_gap * (lvl + 1)
        y_tick = y_bar - tick_h
        ax.plot([x_left, x_left, x_right, x_right], [y_tick, y_bar, y_bar, y_tick],
                color='black', linewidth=0.5)
        ax.text((x_left + x_right) / 2, y_bar + tick_h * 0.4, label,
                ha='center', va='bottom', fontsize=4.3)

# ---- place with a rough first guess, then measure real overflow and correct ----
# (no bbox_inches='tight' at save time -- figure is placed in Illustrator at 100% scale)
guess_left, guess_bottom, guess_top = 0.5, 0.55, 0.35
set_axes_position_inches(fig, ax, left_in=guess_left, top_in=guess_top,
                          width_in=FIG_WIDTH_IN - guess_left - 0.05,
                          height_in=FIG_HEIGHT_IN - guess_top - guess_bottom)

fig.canvas.draw()
renderer = fig.canvas.get_renderer()
tight_bbox = fig.get_tightbbox(renderer)

overflow_left   = max(0, -tight_bbox.x0)
overflow_bottom = max(0, -tight_bbox.y0)
overflow_right  = max(0, tight_bbox.x1 - FIG_WIDTH_IN)
overflow_top    = max(0, tight_bbox.y1 - FIG_HEIGHT_IN)

LEFT_MARGIN_IN   = guess_left   + overflow_left   + 0.03
BOTTOM_MARGIN_IN = guess_bottom + overflow_bottom + 0.03
RIGHT_MARGIN_IN  = 0.05         + overflow_right  + 0.03
TOP_MARGIN_IN    = guess_top    + overflow_top    + 0.03
PLOT_WIDTH_IN    = FIG_WIDTH_IN  - LEFT_MARGIN_IN - RIGHT_MARGIN_IN
PLOT_HEIGHT_IN   = FIG_HEIGHT_IN - TOP_MARGIN_IN  - BOTTOM_MARGIN_IN

set_axes_position_inches(fig, ax, left_in=LEFT_MARGIN_IN, top_in=TOP_MARGIN_IN,
                          width_in=PLOT_WIDTH_IN, height_in=PLOT_HEIGHT_IN)

print(f"margins (in): left={LEFT_MARGIN_IN:.2f} right={RIGHT_MARGIN_IN:.2f} "
      f"top={TOP_MARGIN_IN:.2f} bottom={BOTTOM_MARGIN_IN:.2f}")
print(f"plot area (in): {PLOT_WIDTH_IN:.2f} x {PLOT_HEIGHT_IN:.2f}")

plt.show()

In [ ]:
with PdfPages(PDF_OUT) as pdf:
    pdf.savefig(fig, dpi=450)
plt.close(fig)
print(f'Saved: {os.path.basename(PDF_OUT)}')


## Part 2 — Shared Gamma-frailty Cox model

Model: $h_{ij}(t) = h_0(t)\, z_i \exp(\beta^\top x_{ij})$, where $i$ indexes patients, $j$ indexes
lines within a patient, and $z_i \sim \text{Gamma}(1/\theta, 1/\theta)$ (mean 1, variance $\theta$)
is the patient-level frailty. Estimated via EM (Klein, 1992):

- **E-step:** given current $\beta$ and a Breslow baseline hazard, compute each patient's posterior
  expected frailty $E[z_i \mid \text{data}]$ and $E[\log z_i \mid \text{data}]$.
- **M-step:** refit the Cox partial likelihood with $\log z_i$ as a fixed offset to update $\beta$;
  update $\theta$ by maximizing its profile objective given the current frailty expectations.

Below: the core Breslow-tie partial-likelihood engine (with offset support), then the EM wrapper.


In [ ]:
def _neg_loglik_grad(beta, X_s, offset_s, groups):
    n, p = X_s.shape
    eta = X_s @ beta + offset_s
    w = np.exp(eta)
    S0_suffix = np.cumsum(w[::-1])[::-1]
    S1_suffix = np.cumsum((w[:, None] * X_s)[::-1], axis=0)[::-1]

    first_idx, d_k, _unused1, _unused2, time_code, event_mask_by_row = groups
    S0_at = S0_suffix[first_idx]
    S1_at = S1_suffix[first_idx]

    ev_mask = event_mask_by_row
    sum_eta_events = np.bincount(time_code[ev_mask], weights=eta[ev_mask], minlength=len(d_k))
    sum_x_events = np.zeros((len(d_k), p))
    for j in range(p):
        sum_x_events[:, j] = np.bincount(time_code[ev_mask], weights=X_s[ev_mask, j], minlength=len(d_k))

    has_event = d_k > 0
    ll = np.sum(sum_eta_events[has_event] - d_k[has_event] * np.log(S0_at[has_event]))
    grad = np.sum(sum_x_events[has_event], axis=0) - np.sum(
        d_k[has_event, None] * (S1_at[has_event] / S0_at[has_event, None]), axis=0
    )
    return -ll, -grad


def _build_groups(time_s, event_s):
    uniq_times, first_idx_all, inv = np.unique(time_s, return_index=True, return_inverse=True)
    d_k = np.bincount(inv, weights=event_s, minlength=len(uniq_times))
    return (first_idx_all, d_k, None, None, inv, event_s.astype(bool))


def fit_cox_breslow(X, time, event, offset, beta_init=None):
    order = np.argsort(time, kind='mergesort')
    time_s, event_s, X_s, offset_s = time[order], event[order], X[order], offset[order]
    groups = _build_groups(time_s, event_s)
    beta0 = np.zeros(X.shape[1]) if beta_init is None else beta_init
    res = minimize(_neg_loglik_grad, beta0, args=(X_s, offset_s, groups),
                   jac=True, method='BFGS', options={'gtol': 1e-6, 'maxiter': 200})
    return res, (order, time_s, event_s, X_s, offset_s, groups)


In [ ]:
def fit_gamma_frailty_cox(X_df, time_days, event, cluster_ids, time_scale=30.44,
                          max_iter=150, tol=1e-4, verbose=True):
    """X_df: DataFrame of covariates (already numeric/dummy). Returns (summary_df, theta)."""
    Xmat = X_df.to_numpy(dtype=float)
    time = time_days / time_scale  # months (scale choice only; doesn't affect HRs)
    n, p = Xmat.shape
    clusters, cluster_idx = np.unique(cluster_ids, return_inverse=True)
    n_clusters = len(clusters)

    beta = np.zeros(p)
    theta = 1.0
    z = np.ones(n_clusters)
    offset = np.zeros(n)
    res = None

    for em_iter in range(max_iter):
        res, fitted = fit_cox_breslow(Xmat, time, event, offset, beta_init=beta)
        beta_new = res.x

        order, time_s, event_s, X_s, offset_s, groups = fitted
        eta_s = X_s @ beta_new + offset_s
        w_s = np.exp(eta_s)
        S0_suffix = np.cumsum(w_s[::-1])[::-1]
        first_idx, d_k, _, _, inv, _ = groups
        S0_at = S0_suffix[first_idx]
        has_event = d_k > 0
        dH0 = np.zeros(len(d_k))
        dH0[has_event] = d_k[has_event] / S0_at[has_event]
        cumH0_at_uniq = np.cumsum(dH0)
        cumH0_per_row_sorted = cumH0_at_uniq[inv]
        cumH0_per_row = np.empty(n)
        cumH0_per_row[order] = cumH0_per_row_sorted

        exp_xb = np.exp(Xmat @ beta_new)  # NOT including offset -- z estimated separately
        H_row = cumH0_per_row * exp_xb
        H_i = np.bincount(cluster_idx, weights=H_row, minlength=n_clusters)
        D_i = np.bincount(cluster_idx, weights=event, minlength=n_clusters)

        z_new = np.clip((D_i + 1 / theta) / (H_i + 1 / theta), 1e-6, None)
        Elogz = digamma(D_i + 1 / theta) - np.log(H_i + 1 / theta)

        def negQ(th):
            th = max(th, 1e-6)
            inv_th = 1 / th
            return -np.sum(inv_th * np.log(inv_th) - gammaln(inv_th) + (inv_th - 1) * Elogz - inv_th * z_new)

        theta_res = minimize_scalar(negQ, bounds=(1e-4, 10.0), method='bounded')
        theta_new = theta_res.x

        delta = np.max(np.abs(beta_new - beta)) + abs(theta_new - theta)
        beta, theta, z = beta_new, theta_new, z_new
        offset = np.log(z)[cluster_idx]

        if verbose and (em_iter % 5 == 0 or delta < tol):
            print(f"    EM iter {em_iter + 1:2d}: delta = {delta:.2e}, theta = {theta:.4f}")
        if delta < tol:
            break
    else:
        print(f"    WARNING: did not converge within {max_iter} iterations "
              f"(final delta = {delta:.2e}). Consider raising max_iter.")

    se = np.sqrt(np.diag(res.hess_inv))
    z_stat = beta / se
    p_val = 2 * (1 - norm.cdf(np.abs(z_stat)))

    summary = pd.DataFrame({
        'covariate': X_df.columns,
        'beta': beta, 'se': se,
        'HR': np.exp(beta),
        'HR_ci_lower': np.exp(beta - 1.96 * se),
        'HR_ci_upper': np.exp(beta + 1.96 * se),
        'p_value': p_val,
    })
    return summary, theta


## Build the covariate-complete cohort and design matrix (shared across all 6 models)

Restricted to rows with clean sex (`Male`/`Female`), non-missing `age_at_lot_start`, and a usable
`cancer_type` (excluding the `Multiple Cancer Type Patient` catch-all). Cancer type is collapsed to
its top 10 categories + `Other`. Reference levels: LOT 1, Male, `Other` cancer type.


In [ ]:
def build_design_matrix(df):
    d = df.copy()
    d['sex_clean'] = d['sex'].astype(str).str.strip().str.capitalize()
    d = d[d['sex_clean'].isin(['Male', 'Female'])].copy()
    d = d[d['cancer_type'].notna() & (d['cancer_type'] != '') &
          (d['cancer_type'] != 'Multiple Cancer Type Patient')].copy()
    d['age_at_lot_start'] = pd.to_numeric(d['age_at_lot_start'], errors='coerce')
    d = d[d['age_at_lot_start'].notna()].copy()
    d = d.reset_index(drop=True)

    top10 = d['cancer_type'].value_counts().head(10).index.tolist()
    d['cancer_type_grp'] = np.where(d['cancer_type'].isin(top10), d['cancer_type'], 'Other')

    lot_dummies = pd.get_dummies(d['lot_group'], prefix='LOT').astype(int)
    lot_dummies = lot_dummies.drop(columns=['LOT_1'])  # reference = LOT 1

    sex_dummy = (d['sex_clean'] == 'Female').astype(int).rename('sex_Female')  # ref = Male

    ct_dummies = pd.get_dummies(d['cancer_type_grp'], prefix='cancer').astype(int)
    ref_col = 'cancer_Other' if 'cancer_Other' in ct_dummies.columns else ct_dummies.columns[0]
    ct_dummies = ct_dummies.drop(columns=[ref_col])  # reference = Other (or fallback)

    age_centered = (d['age_at_lot_start'] - d['age_at_lot_start'].mean()).rename('age_centered')

    X = pd.concat([lot_dummies, age_centered, sex_dummy, ct_dummies], axis=1)
    return d, X

cohort_df, X_full = build_design_matrix(all_lots)
print(f'{len(cohort_df):,} LOT-level rows with complete covariates for the frailty models')
print(f'Design matrix columns: {list(X_full.columns)}')


## Time-to-first-AE for each row (per toxicity)

Same logic as the KM part: find each row's own first-AE time within `[lot_start, lot_start + censor_days]`.


In [ ]:
def compute_time_event(df, tox):
    if tox not in batch_df.columns:
        return np.zeros(len(df)), np.zeros(len(df))
    bsub = batch_df[batch_df[tox] == 1][['mrn', 'window_start']]
    bsub_by_mrn = dict(list(bsub.groupby('mrn')))
    times = np.empty(len(df))
    events = np.empty(len(df))
    for i, row in enumerate(df.itertuples()):
        mrn, lot_start, censor_d = row.mrn, row.lot_start, row.censor_days
        first_ae_days = None
        if mrn in bsub_by_mrn:
            recs = bsub_by_mrn[mrn]
            days = (recs['window_start'] - lot_start).dt.days
            in_window = days[(days >= 0) & (days <= censor_d)]
            if len(in_window) > 0:
                first_ae_days = float(in_window.min())
        if first_ae_days is not None:
            times[i] = first_ae_days
            events[i] = 1
        else:
            times[i] = censor_d
            events[i] = 0
    return times, events


In [ ]:
# ---------------------------------------------------------------------------
# Export analysis-ready CSV for Cox modeling in R (coxme)
# Includes treatment flags for consistency with main figure adjustments
# ---------------------------------------------------------------------------
R_EXPORT_PATH = os.path.join(RESULTS_DIR, 'LOT_prevalence_S2E_cox_input.csv')

# Helper to convert flag values
def _flag_on_export(x):
    if pd.isna(x):
        return 0
    if isinstance(x, bool):
        return int(x)
    if isinstance(x, (int, float)):
        return int(x == 1)
    return int(str(x).strip().lower() in ('1', 'true', 'yes'))

# Start with base columns
export_df = cohort_df[['mrn', 'lot_clean', 'lot_group', 'age_at_lot_start',
                        'sex_clean', 'cancer_type_grp']].copy()
export_df = export_df.rename(columns={'lot_clean': 'lot'})

# Check which treatment columns exist in cohort_df
treatment_cols = ['contains_ctla4_immuno', 'contains_ctla4', 'contains_non_ctla4_immuno', 
                  'contains_pd1', 'contains_chemo', 'contains_hormone', 
                  'contains_biologic', 'contains_targeted']
available_cols = [c for c in treatment_cols if c in cohort_df.columns]
print(f"Available treatment columns in cohort_df: {available_cols}")

# Compute combined immuno flags
if 'contains_non_ctla4_immuno' in cohort_df.columns and 'contains_pd1' in cohort_df.columns:
    export_df['has_pd1_flag'] = (cohort_df['contains_non_ctla4_immuno'].apply(_flag_on_export) |
                                  cohort_df['contains_pd1'].apply(_flag_on_export)).astype(int)
else:
    export_df['has_pd1_flag'] = 0
    print("Warning: PD1 columns not found, setting has_pd1_flag to 0")

if 'contains_ctla4_immuno' in cohort_df.columns and 'contains_ctla4' in cohort_df.columns:
    export_df['has_ctla4_flag'] = (cohort_df['contains_ctla4_immuno'].apply(_flag_on_export) |
                                    cohort_df['contains_ctla4'].apply(_flag_on_export)).astype(int)
else:
    export_df['has_ctla4_flag'] = 0
    print("Warning: CTLA4 columns not found, setting has_ctla4_flag to 0")

# Convert other treatment flags
for col in ['contains_chemo', 'contains_hormone', 'contains_biologic', 'contains_targeted']:
    if col in cohort_df.columns:
        export_df[col] = cohort_df[col].apply(_flag_on_export).astype(int)
    else:
        export_df[col] = 0
        print(f"Warning: {col} not found, setting to 0")

# Add toxicity time/event columns
for tox in TOXICITY_COLUMNS:
    times, events = compute_time_event(cohort_df, tox)
    export_df[f'{tox}_time_days'] = times
    export_df[f'{tox}_event'] = events.astype(int)

export_df.to_csv(R_EXPORT_PATH, index=False)
print(f'\nSaved: {os.path.basename(R_EXPORT_PATH)}  ({len(export_df):,} rows)')
print(f'Treatment flag columns: has_pd1_flag, has_ctla4_flag, contains_chemo, contains_hormone, contains_biologic, contains_targeted')
# Preview without 'mrn' so no patient identifiers are stored in notebook output.
export_df.drop(columns=['mrn']).head()